# routes.handlers

> Response builder functions for virtual collection navigation (Tier 1 API).

In [ ]:
#| default_exp routes.handlers

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Any, Callable, List, Tuple

from fasthtml.common import Hidden

from cjm_fasthtml_virtual_collection.core.models import (
    VirtualCollectionConfig, VirtualCollectionState, VirtualCollectionUrls,
)
from cjm_fasthtml_virtual_collection.core.html_ids import VirtualCollectionHtmlIds
from cjm_fasthtml_virtual_collection.core.windowing import (
    navigate, navigate_cursor, clamp_window_start, compute_window,
)
from cjm_fasthtml_virtual_collection.components.table import render_slot, render_table_rows
from cjm_fasthtml_virtual_collection.components.footer import render_footer

## Response Builders

In [ ]:
#| export
def _render_window_start_oob(
    state: VirtualCollectionState,     # Current state
    ids: VirtualCollectionHtmlIds,     # HTML IDs
) -> Hidden:  # Hidden input with OOB swap
    """Render OOB hidden input carrying the current window_start for JS thumb positioning."""
    return Hidden(
        value=str(state.window_start),
        id=ids.window_start_input,
        name="window_start",
        hx_swap_oob="outerHTML",
    )


def build_nav_response(
    items: list,                            # Full item list
    state: VirtualCollectionState,          # Current state (already mutated)
    config: VirtualCollectionConfig,        # Collection config
    ids: VirtualCollectionHtmlIds,          # HTML IDs
    render_cell: Callable,                  # Consumer cell render callback
    focus_url: str = "",                    # URL for click-to-focus
) -> Tuple:  # OOB elements (slot OOBs + footer + window_start input)
    """Build OOB response for navigation: all visible slots + footer + window_start."""
    render_start, render_end = compute_window(
        state.window_start, state.visible_rows, state.total_items
    )

    slot_oobs = [
        render_slot(
            slot_index=slot_idx, item=items[item_idx], item_index=item_idx,
            config=config, state=state, ids=ids, render_cell=render_cell,
            oob=True, focus_url=focus_url,
        )
        for slot_idx, item_idx in enumerate(range(render_start, render_end))
    ]

    return tuple(slot_oobs) + (
        render_footer(state, ids, oob=True),
        _render_window_start_oob(state, ids),
    )

In [ ]:
#| export
def build_cursor_move_response(
    old_cursor: int,                        # Previous cursor index
    items: list,                            # Full item list
    state: VirtualCollectionState,          # Current state (cursor already updated)
    config: VirtualCollectionConfig,        # Collection config
    ids: VirtualCollectionHtmlIds,          # HTML IDs
    render_cell: Callable,                  # Consumer cell render callback
    focus_url: str = "",                    # URL for click-to-focus
) -> Tuple:  # OOB elements (affected slot OOBs + footer + window_start input)
    """Build OOB response for cursor-only move: swap just the affected slots."""
    oob_parts = []

    # Re-render old cursor slot (remove highlight)
    if old_cursor != state.cursor_index:
        old_slot = old_cursor - state.window_start
        oob_parts.append(render_slot(
            slot_index=old_slot, item=items[old_cursor], item_index=old_cursor,
            config=config, state=state, ids=ids, render_cell=render_cell,
            oob=True, focus_url=focus_url,
        ))

    # Re-render new cursor slot (add highlight)
    new_slot = state.cursor_index - state.window_start
    oob_parts.append(render_slot(
        slot_index=new_slot, item=items[state.cursor_index], item_index=state.cursor_index,
        config=config, state=state, ids=ids, render_cell=render_cell,
        oob=True, focus_url=focus_url,
    ))

    oob_parts.append(render_footer(state, ids, oob=True))
    oob_parts.append(_render_window_start_oob(state, ids))

    return tuple(oob_parts)

## Navigation Handlers (Tier 1)

In [ ]:
#| export
def _is_cursor_visible(
    state: VirtualCollectionState,  # Current state
) -> bool:  # Whether cursor is within the visible window
    """Check if the cursor index is within the current visible window."""
    if state.cursor_index < 0:
        return False
    return state.window_start <= state.cursor_index < state.window_start + state.visible_rows


def handle_navigate(
    direction: str,                         # 'up', 'down', 'page_up', 'page_down', 'first', 'last'
    items: list,                            # Full item list
    state: VirtualCollectionState,          # Current state (mutated in place)
    config: VirtualCollectionConfig,        # Collection config
    ids: VirtualCollectionHtmlIds,          # HTML IDs
    render_cell: Callable,                  # Consumer cell render callback
    focus_url: str = "",                    # URL for click-to-focus
) -> Tuple:  # OOB elements
    """Navigate in a direction. Mutates state in place."""
    if direction in ('up', 'down'):
        # Reset cursor to top-most visible row if off-screen
        if not _is_cursor_visible(state):
            state.cursor_index = state.window_start
            return build_nav_response(items, state, config, ids, render_cell, focus_url)

        old_cursor = state.cursor_index
        new_cursor, new_window_start, window_changed = navigate_cursor(
            state.cursor_index, direction, state.window_start,
            state.visible_rows, state.total_items,
        )
        state.cursor_index = new_cursor
        state.window_start = new_window_start

        if window_changed:
            return build_nav_response(items, state, config, ids, render_cell, focus_url)
        else:
            return build_cursor_move_response(old_cursor, items, state, config, ids, render_cell, focus_url)
    else:
        state.window_start = navigate(
            state.window_start, direction, state.visible_rows, state.total_items
        )
        return build_nav_response(items, state, config, ids, render_cell, focus_url)

In [ ]:
#| export
def handle_navigate_to_index(
    target_index: int,                      # Target window_start
    items: list,                            # Full item list
    state: VirtualCollectionState,          # Current state (mutated in place)
    config: VirtualCollectionConfig,        # Collection config
    ids: VirtualCollectionHtmlIds,          # HTML IDs
    render_cell: Callable,                  # Consumer cell render callback
    focus_url: str = "",                    # URL for click-to-focus
) -> Tuple:  # OOB elements
    """Navigate to a specific index. Mutates state.window_start in place."""
    state.window_start = clamp_window_start(
        target_index, state.visible_rows, state.total_items
    )
    return build_nav_response(items, state, config, ids, render_cell, focus_url)

In [ ]:
#| export
def _build_container_response(
    items: list,                            # Full item list
    state: VirtualCollectionState,          # Current state
    config: VirtualCollectionConfig,        # Collection config
    ids: VirtualCollectionHtmlIds,          # HTML IDs
    render_cell: Callable,                  # Consumer cell render callback
    focus_url: str = "",                    # URL for click-to-focus
) -> Tuple:  # OOB elements (container + footer + window_start input)
    """Build OOB response that replaces the entire rows container with new slots."""
    rows_oob = render_table_rows(items, config, state, ids, render_cell, focus_url=focus_url)
    rows_oob.attrs["hx-swap-oob"] = "outerHTML"
    return (
        rows_oob,
        render_footer(state, ids, oob=True),
        _render_window_start_oob(state, ids),
    )


def handle_update_viewport(
    visible_rows: int,                      # New visible row count
    items: list,                            # Full item list
    state: VirtualCollectionState,          # Current state (mutated in place)
    config: VirtualCollectionConfig,        # Collection config
    ids: VirtualCollectionHtmlIds,          # HTML IDs
    render_cell: Callable,                  # Consumer cell render callback
    is_auto: bool = True,                   # Whether from auto-fit
    focus_url: str = "",                    # URL for click-to-focus
) -> Tuple:  # OOB elements
    """Update viewport with new row count. Mutates state in place."""
    visible_rows = max(1, visible_rows)
    state.visible_rows = visible_rows
    state.is_auto_mode = is_auto
    state.window_start = clamp_window_start(
        state.window_start, state.visible_rows, state.total_items
    )
    # Replace entire container — slot count changed
    return _build_container_response(items, state, config, ids, render_cell, focus_url)

In [ ]:
#| export
def handle_focus_row(
    row_index: int,                         # Row index to focus
    items: list,                            # Full item list
    state: VirtualCollectionState,          # Current state (mutated in place)
    config: VirtualCollectionConfig,        # Collection config
    ids: VirtualCollectionHtmlIds,          # HTML IDs
    render_cell: Callable,                  # Consumer cell render callback
    focus_url: str = "",                    # URL for click-to-focus
) -> Tuple:  # OOB elements (affected slot OOBs + footer + window_start input)
    """Move cursor to a specific row via click/tap. Mutates state in place."""
    old_cursor = state.cursor_index
    state.cursor_index = max(0, min(row_index, state.total_items - 1))
    return build_cursor_move_response(old_cursor, items, state, config, ids, render_cell, focus_url)

## Tests

In [ ]:
from dataclasses import dataclass
from fasthtml.common import to_xml, Span
from cjm_fasthtml_virtual_collection.core.models import (
    VirtualCollectionConfig, VirtualCollectionState, ColumnDef,
)
from cjm_fasthtml_virtual_collection.core.html_ids import VirtualCollectionHtmlIds

@dataclass
class Item:
    name: str

test_items = [Item(f"file_{i}.txt") for i in range(50)]
columns = (ColumnDef(key="name", header="Name", width="1fr"),)
config = VirtualCollectionConfig(prefix="th", columns=columns)
ids = VirtualCollectionHtmlIds(prefix="th")

def test_render_cell(item, ctx):
    return Span(item.name)

In [ ]:
# Test handle_navigate down — cursor moves within window (no scroll)
state = VirtualCollectionState(total_items=50, visible_rows=5, window_start=0, cursor_index=0)
result = handle_navigate("down", test_items, state, config, ids, test_render_cell)
assert state.cursor_index == 1
assert state.window_start == 0

# Response: 2 slot OOBs + footer + window_start
assert len(result) == 4
old_slot_html = to_xml(result[0])
assert 'id="th-slot-0"' in old_slot_html
assert 'hx-swap-oob="innerHTML"' in old_slot_html
new_slot_html = to_xml(result[1])
assert 'id="th-slot-1"' in new_slot_html
assert 'hx-swap-oob="innerHTML"' in new_slot_html
# Inner row has cursor highlight
assert 'bg-base-300' in new_slot_html
assert 'bg-base-300' not in old_slot_html
print("handle_navigate down (cursor move) test passed")

handle_navigate down (cursor move) test passed


In [ ]:
# Test handle_navigate down — cursor at bottom edge triggers scroll
state = VirtualCollectionState(total_items=50, visible_rows=5, window_start=0, cursor_index=4)
result = handle_navigate("down", test_items, state, config, ids, test_render_cell)
assert state.cursor_index == 5
assert state.window_start == 1

# Response: 5 slot OOBs + footer + window_start = 7 parts
assert len(result) == 7
# All slots have innerHTML OOB
for i in range(5):
    slot_html = to_xml(result[i])
    assert f'id="th-slot-{i}"' in slot_html
    assert 'hx-swap-oob="innerHTML"' in slot_html
# Cursor is at bottom slot (slot 4 = item 5)
assert 'bg-base-300' in to_xml(result[4])
assert 'bg-base-300' not in to_xml(result[0])
print("handle_navigate down (scroll at edge) test passed")

# Test handle_navigate up — clamped at top
state = VirtualCollectionState(total_items=50, visible_rows=5, window_start=0, cursor_index=0)
result = handle_navigate("up", test_items, state, config, ids, test_render_cell)
assert state.cursor_index == 0
assert state.window_start == 0
print("handle_navigate up (clamped at top) test passed")

handle_navigate down (scroll at edge) test passed
handle_navigate up (clamped at top) test passed


In [ ]:
# Test handle_navigate_to_index
state = VirtualCollectionState(total_items=50, visible_rows=5, window_start=0)
result = handle_navigate_to_index(25, test_items, state, config, ids, test_render_cell)
assert state.window_start == 25

rows_html = to_xml(result[0])
assert 'file_25.txt' in rows_html
print("handle_navigate_to_index test passed")

handle_navigate_to_index test passed


In [ ]:
# Test handle_update_viewport — replaces entire container
state = VirtualCollectionState(total_items=50, visible_rows=5, window_start=46)
result = handle_update_viewport(10, test_items, state, config, ids, test_render_cell)
assert state.visible_rows == 10
assert state.window_start == 40  # re-clamped: 50 - 10 = 40

# Response is container outerHTML + footer + window_start
assert len(result) == 3
container_html = to_xml(result[0])
assert 'hx-swap-oob="outerHTML"' in container_html
assert f'id="{ids.rows}"' in container_html
# Contains 10 slots
assert 'id="th-slot-0"' in container_html
assert 'id="th-slot-9"' in container_html
assert 'id="th-slot-10"' not in container_html
print("handle_update_viewport test passed")

handle_update_viewport test passed


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()